In [1]:
import pandas as pd
import sqlite3

/root/.venv/lib/python3.12/site-packages/numpy/_core/getlimits.py:551: UserWarning: Signature b'\x00\xd0\xcc\xcc\xcc\xcc\xcc\xcc\xfb\xbf\x00\x00\x00\x00\x00\x00' for <class 'numpy.longdouble'> does not match any known type: falling back to type probe function.
This warnings indicates broken support for the dtype!
  machar = _get_machar(dtype)


### Создадим подключение к БД с помощью sqlite3

In [2]:
conn = sqlite3.connect('../data/checking-logs.sqlite')

### Используя только один запрос для каждой из групп, создайте два фрейма данных: test_results и control_results со столбцами time и avg_diff и только двумя строками

* time должно иметь значения: after и before
* avg_diff содержит среднюю дельту среди всех пользователей за период времени до того, как каждый из них впервые посетил страницу, и после
* учитывайте только тех пользователей, у которых есть наблюдения до и после
* мы по-прежнему не используем 'project1'

In [3]:
query = """
SELECT time,
       AVG(delta) AS avg_diff
FROM (SELECT uid,
             CAST(((JulianDay(datetime(d.deadlines, 'unixepoch')) -
                   JulianDay(t.first_commit_ts)) * 24) AS Integer) AS delta,
             CASE WHEN t.first_commit_ts < t.first_view_ts THEN 'before'
                                                           ELSE 'after'
                                                           END AS time
      FROM test AS t LEFT JOIN deadlines AS d on t.labname = d.labs
      WHERE labname != 'project1')
WHERE uid in (SELECT uid
              FROM (SELECT uid,
                           CASE WHEN t.first_commit_ts < t.first_view_ts THEN 'before'
                                                                         ELSE 'after'
                                                                         END as time
                    FROM test as t LEFT JOIN deadlines AS d ON t.labname=d.labs
                    WHERE labname != 'project1')
               GROUP BY uid
               HAVING COUNT(DISTINCT time)=2)
GROUP BY time
"""
test_results = pd.io.sql.read_sql(query, conn)
test_results

,time,avg_diff
0,after,104.6000
1,before,60.5625


In [4]:
query = """
SELECT time,
       AVG(delta) AS avg_diff
FROM (SELECT uid,
             CAST(((JulianDay(datetime(d.deadlines, 'unixepoch')) -
                   JulianDay(c.first_commit_ts)) * 24) AS Integer) AS delta,
             CASE WHEN c.first_commit_ts < c.first_view_ts THEN 'before'
                                                           ELSE 'after'
                                                           END AS time
      FROM control AS c LEFT JOIN deadlines AS d on c.labname = d.labs
      WHERE labname != 'project1')
WHERE uid in (SELECT uid
              FROM (SELECT uid,
                           CASE WHEN c.first_commit_ts < c.first_view_ts THEN 'before'
                                                                         ELSE 'after'
                                                                         END as time
                    FROM control as c LEFT JOIN deadlines AS d ON c.labname=d.labs
                    WHERE labname != 'project1')
               GROUP BY uid
               HAVING COUNT(DISTINCT time)=2)
GROUP BY time
"""
control_results = pd.io.sql.read_sql(query, conn)
control_results

,time,avg_diff
0,after,117.636364
1,before,99.464286


### Закроем соединение с базой данных

In [5]:
conn.close()

Заметим что для выборки test, значение поменялось гораздо сильнее чем для выборки control. Значит наша гипотеза верна.